In [ ]:
import os
import glob
import shutil
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df=pd.read_csv('/content/drive/My Drive/Colab Notebooks/image topic/image topic only camptioning.csv', encoding='utf-8') #导入主题分类数据
df.head()

,Unnamed: 0.1,Unnamed: 0,Image,Text,topic,probabilities
0,0,0,@ngeonmak2_video_7086687826590403867.mp4_2.jpg,there is a picture of a poem written in thai o...,0,NaN
1,1,1,@nexarson_video_7088496739224669482.mp4_5.jpg,arafed person laughing and laughing with a tex...,1,NaN
2,2,2,@nexarson_video_7088496739224669482.mp4_4.jpg,arafed person with a funny expression on her face,1,NaN
3,3,3,@nexarson_video_7088496739224669482.mp4_2.jpg,laptop computer with a message on the screen a...,5,NaN
4,4,4,@ngeonmak2_video_7086687826590403867.mp4_1.jpg,there is a poem written on a wall with a pictu...,0,NaN


In [ ]:
df = df[df['topic'] != -1]
df.shape

(6359, 9)

In [ ]:
# 按topic和prob排序
#df_sorted = df.sort_values(['topic', 'prob'], ascending=[True, False])
# 用groupby函数获取每个class分类中degree排名前50的数据
#df_top_50 = df_sorted.groupby('topic').head(50)
#df_top_50.head()

,Unnamed: 0,Image,Text,cartoon,text,background,content,topic,prob
7,7,@news.com.au_video_7157913634226654466.mp4_4.jpg,arafed woman in orange vests and safety vest s...,NaN,text,natural landscape,arafed person in orange vests and safety vest ...,0,1.0
8,8,@news.com.au_video_7157913634226654466.mp4_5.jpg,arafed man and woman sitting in front of a pai...,NaN,text,natural landscape,arafed person and person sitting in front of a...,0,1.0
12,12,@liam.out.loud_video_7155586485599685893.mp4_4...,there is a man in a suit and tie with a red ba...,NaN,text,NaN,there is a person in a suit and tie with a red...,0,1.0
36,36,@liilneri_video_7211915760480832773.mp4_4.jpg,there is a woman sitting on a bench in a park,NaN,NaN,outdoor,there is a person sitting on a bench in a park...,0,1.0
50,50,@lilwolf727_video_7206018302018243886.mp4_1.jpg,there is a man standing in the middle of a str...,NaN,NaN,NaN,there is a person standing in the middle of a ...,0,1.0


In [ ]:
#随机选择每个topic的图片

import random
# 新的DataFrame用于存储每个topic随机选择的50个images
df_top_50 = pd.DataFrame()

# 对每个topic值进行处理
for topic_value in df['topic'].unique():
    # 获取该topic值对应的所有images
    images_for_topic = df[df['topic'] == topic_value]['Image']

    # 随机选择10个images（如果images数量小于10，则选择全部）
    selected_images_for_topic = random.sample(list(images_for_topic), min(len(images_for_topic), 20))  #20shi

    # 将选择的images添加到新的DataFrame中
    df_top_50 = df_top_50.append(pd.DataFrame({'images': selected_images_for_topic, 'topic': topic_value}))

# 重置索引
df_top_50.reset_index(drop=True, inplace=True)

<ipython-input-4-10e2d1f029a9>:16: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_top_50 = df_top_50.append(pd.DataFrame({'images': selected_images_for_topic, 'topic': topic_value}))
<ipython-input-4-10e2d1f029a9>:16: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_top_50 = df_top_50.append(pd.DataFrame({'images': selected_images_for_topic, 'topic': topic_value}))
<ipython-input-4-10e2d1f029a9>:16: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_top_50 = df_top_50.append(pd.DataFrame({'images': selected_images_for_topic, 'topic': topic_value}))
<ipython-input-4-10e2d1f029a9>:16: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  df_top_5

## core images

In [ ]:
# Step 2: 将Google Drive中的文件路径添加到系统路径中
import sys
sys.path.append('/content/drive/My Drive/Colab Notebooks/image topic/')

In [ ]:
import os, sys, shutil
import numpy as np
import pandas as pd
import ypoften as of
import stat

In [ ]:
def save_img(imges, label):
    imgpath1='/content/drive/My Drive/Colab Notebooks/image_frames_data/'+imges #所有图片的文件路径
    imgpath2 = os.path.join('/content/drive/My Drive/Colab Notebooks/image topic/', 'BERTopic10-only camptioning', str(label),imges) #需要保存到的路径
    of.create_path(imgpath2)
    shutil.copy2(imgpath1, imgpath2)

In [ ]:
df_top_50.apply(lambda row: save_img(row['images'],row['topic']),axis=1)

0      None
1      None
2      None
3      None
4      None
       ... 
195    None
196    None
197    None
198    None
199    None
Length: 200, dtype: object

## visuliazing

In [ ]:
import os, sys, string, glob
from PIL import Image, ImageDraw, ImageFont
from natsort import natsorted
import numpy as np
import ypoften as of

In [ ]:
def fill_square(im, tbw):
    size = (tbw, tbw)
    bg = Image.new('RGB', size, "white")  # create a background image
    im.thumbnail(size, Image.ANTIALIAS)
    w, h = im.size
    bg.paste(im, (int((size[0]-w)/2), int((size[1]-h)/2)))
    return(bg)

In [ ]:
w_sq = 300 # width/height of image
w_clusterid = 60 # width of text for cluster id
h_imgid = 60 # height of gap
#fntcluster = ImageFont.truetype('/content/drive/My Drive/Colab Notebooks/image topic/Arial.ttf',60)
fntcluster = ImageFont.load_default()

In [ ]:
nclu=10 #修改聚类个数
ncol = 20 # how many images to show in each row
nrow = 1
nimg = ncol * nrow # total number of image in each cluster to show
large_w = w_sq*ncol + w_clusterid # total width of the entire image
large_h = (w_sq + h_imgid) * nrow * nclu # total height of the entire image #修改序列
large = Image.new('RGB', (large_w, large_h), "white")

for label in range(0, nclu):
    print('-'* 10, nclu, label)
    imgpathfolder = os.path.join('/content/drive/My Drive/Colab Notebooks/image topic/', 'BERTopic10-only camptioning', str(label),'') #修改文件名，此处路径与imgpath2相同
    imgfiles = glob.glob(imgpathfolder + "*.jpg")
    imgfiles = natsorted(imgfiles)[:nimg]

    d = ImageDraw.Draw(large)
    d.text((0, label * nrow * (w_sq + h_imgid) + 15), str(label+1), font=fntcluster, fill=(0, 0, 0))

    for j, imgfile in enumerate(imgfiles):
        jw = j%ncol; jh = j//ncol
        img = Image.open(imgfile)
        im_sq = fill_square(img, w_sq)
        im_sq_x = jw*w_sq + w_clusterid
        im_sq_y = h_imgid + jh* (w_sq + h_imgid) + label * nrow * (w_sq + h_imgid)
        large.paste(im_sq, (im_sq_x, im_sq_y))

large_savepath = os.path.join('/content/drive/My Drive/Colab Notebooks/image topic/BERTopic10-only camptioning/grid/abstract.png') #修改文件名，将缩略图保存到想保存的路径
of.create_path(large_savepath)
large.save(large_savepath)

---------- 10 0


<ipython-input-11-92eab113e7db>:4: DeprecationWarning: ANTIALIAS is deprecated and will be removed in Pillow 10 (2023-07-01). Use LANCZOS or Resampling.LANCZOS instead.
  im.thumbnail(size, Image.ANTIALIAS)


---------- 10 1
---------- 10 2
---------- 10 3
---------- 10 4
---------- 10 5
---------- 10 6
---------- 10 7
---------- 10 8
---------- 10 9
